<a href="https://colab.research.google.com/github/shreeyarawal/mac237-labs/blob/main/lab4/MAC237_LAB4_S16_SQL_Injection_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 4, Session 16: SQL injection, shown and then defended

**Wednesday, November 11, 2026. This session runs in Google Colab, in a browser.**

B124 is not available to us on Wednesday evenings, so this session is a notebook you run on
any computer with a browser. It uses only the Python 3 standard library, needs no
administrative rights, and reaches no network: the database lives in memory and is built
from your own token.

**Nothing in this notebook contacts any host outside it.** The whole bug is one operator.
Build a query by gluing strings together and the user gets to write SQL; pass the values as
parameters and the user gets to write values, and nothing else. You will run both and compare
the rows they return.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token must be visible in the first
and the last screenshot of this session.

In [ ]:
import hashlib, datetime

sid = '24621670'          # put your own student ID here, in quotes
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 4 token:', TOKEN)

MAC237 Lab 4 token: 21d7ec28ef7732ea


## Section 3. SQL injection, exploited then defended

The cell below builds a three-row users table from your token, then checks a login two ways.
`login_concat` builds the SQL by joining strings, the way far too much code still does.
`login_param` sends the same query with placeholders and passes the values separately. Watch
what each does with an ordinary wrong password and with a crafted input that closes the
quote and adds `OR '1'='1`.

In [ ]:
import sqlite3

# A tiny users table, personalized so the rows are yours.
seed = int(TOKEN[:6], 16)
users = [(1, 'alice', TOKEN[:8]),
         (2, 'bob',   TOKEN[8:16]),
         (3, 'carol', hex(seed)[2:10])]
db = sqlite3.connect(':memory:')
db.execute('CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT, pw TEXT)')
db.executemany('INSERT INTO users VALUES (?, ?, ?)', users)
db.commit()

def login_concat(name, pw):
    # VULNERABLE: the values are pasted straight into the SQL text.
    sql = "SELECT id, name FROM users WHERE name = '" + name + "' AND pw = '" + pw + "'"
    return sql, db.execute(sql).fetchall()

def login_param(name, pw):
    # SAFE: the SQL is fixed; the values arrive separately and are never parsed as SQL.
    sql = 'SELECT id, name FROM users WHERE name = ? AND pw = ?'
    return sql, db.execute(sql, (name, pw)).fetchall()

print('token:', TOKEN)
print()
good_pw = TOKEN[:8]
injection = "' OR '1'='1"

for label, fn in (('concatenated', login_concat), ('parameterized', login_param)):
    print(label)
    for case, name, pw in (('correct password', 'alice', good_pw),
                           ('wrong password',   'alice', 'nope'),
                           ('injection attempt', 'alice', injection)):
        sql, rows = fn(name, pw)
        print(f'  {case:18} -> {len(rows)} row(s): {rows}')
        if case == 'injection attempt':
            print(f'    SQL was: {sql}')
    print()

_, concat_inj = login_concat('alice', injection)
_, param_inj  = login_param('alice', injection)
assert len(concat_inj) == 3, 'the concatenated query should leak all rows on injection'
assert len(param_inj) == 0, 'the parameterized query should return nothing on injection'
print('  CHECK PASS: the concatenated query is fooled into returning every row')
print('  CHECK PASS: the parameterized query returns nothing to the same input')

token: 21d7ec28ef7732ea

concatenated
  correct password   -> 1 row(s): [(1, 'alice')]
  wrong password     -> 0 row(s): []
  injection attempt  -> 3 row(s): [(1, 'alice'), (2, 'bob'), (3, 'carol')]
    SQL was: SELECT id, name FROM users WHERE name = 'alice' AND pw = '' OR '1'='1'

parameterized
  correct password   -> 1 row(s): [(1, 'alice')]
  wrong password     -> 0 row(s): []
  injection attempt  -> 0 row(s): []
    SQL was: SELECT id, name FROM users WHERE name = ? AND pw = ?

  CHECK PASS: the concatenated query is fooled into returning every row
  CHECK PASS: the parameterized query returns nothing to the same input


## What goes in the report

Both `CHECK PASS` lines must appear above.

1. Explain what a parameterized query does that escaping the input does not, in terms of when
   the statement is parsed and when the values arrive.
2. Read the printed SQL for the injected case. Point to the exact character where the input
   stops being data and starts being SQL.
3. The concatenated check returned all three users to someone who supplied one wrong password.
   Say what that would mean for a real login form, and name one thing besides parameterized
   queries that a login should also do so that one leaked query is not the whole breach.
4. Cite CTPE Chapter 5 on software security.

**Screenshot** `lab4_s16_sqli_before_after.png`, showing all six result lines and the printed
SQL for the injected case, with your token and your Google account name in the browser.

**Save and sign out.** Save the notebook into `lab4` with File, then Save a copy in GitHub;
choose Runtime, then Disconnect and delete runtime; sign out of Google on a shared computer;
and capture `lab4_s16_logout.png` with the token on screen. Upload the two screenshots to
`lab4` on github.com.